In [1]:
import os
from pathlib import Path
import json
from pprint import pprint
from collections import Counter

from colorama import Fore, Style

from datasets import load_dataset, DatasetDict
from huggingface_hub import dataset_info

from tqdm import tqdm

In [2]:
from datasets import load_dataset

ds_path = "nvidia/Nemotron-PII"
ds = load_dataset(ds_path)

In [3]:
DATA_DIR = Path(os.getcwd()).resolve().parent.parent / "data"
LABEL_INFO_PATH = DATA_DIR/"nemotron-label_info.json"
DATASET_PATH = DATA_DIR/"nemotron-pii"
print(DATASET_PATH)
print(LABEL_INFO_PATH)

/home/zelluzy/Desktop/pii_redaction_api/data/nemotron-pii
/home/zelluzy/Desktop/pii_redaction_api/data/nemotron-label_info.json


In [4]:
info = dataset_info(ds_path)
print(info.card_data)

language:
- en
license: cc-by-4.0
size_categories:
- 100K<n<1M
task_categories:
- token-classification
tags:
- datadesigner
- pii
- privacy
- data-masking
- synthetic-data
- named-entity-recognition
- nvidia
- nemotron
- personas


In [5]:
pprint(ds["train"].features)

{'document_description': Value('string'),
 'document_format': Value('string'),
 'document_type': Value('string'),
 'domain': Value('string'),
 'locale': Value('string'),
 'spans': Value('string'),
 'text': Value('string'),
 'text_tagged': Value('string'),
 'uid': Value('string')}


In [6]:
for split in ds:
    print(split)

train
test


In [7]:
print(f"length of full dataset: {sum([len(ds[split]) for split in ds]):,}")

length of full dataset: 200,000


In [8]:
pprint(ds["train"][0])

{'document_description': 'This document, typically found in the Identity '
                         'Verification Services and Financial Services '
                         'domains, is an unstructured application form '
                         'featuring handwritten or typed responses, including '
                         'fields for personal identification details such as '
                         'name, date of birth, and address, along with '
                         'financial information like income and employment '
                         'status, and often includes scanned or attached '
                         'copies of identification documents and signatures.',
 'document_format': 'unstructured',
 'document_type': 'Financial Services Application',
 'domain': 'Identity Verification Services',
 'locale': 'us',
 'spans': "[{'start': 3, 'end': 8, 'text': 'Jason', 'label': 'first_name'}, "
          "{'start': 76, 'end': 86, 'text': '1987-05-22', 'label': "
          "'date_of

In [9]:
train_mask_feat = ds["train"]["spans"]
example = next(iter(train_mask_feat))
pprint(example) # validate

("[{'start': 3, 'end': 8, 'text': 'Jason', 'label': 'first_name'}, {'start': "
 "76, 'end': 86, 'text': '1987-05-22', 'label': 'date_of_birth'}, {'start': "
 "98, 'end': 123, 'text': '87 Avenida De La Estrella', 'label': "
 "'street_address'}, {'start': 139, 'end': 148, 'text': 'full-time', 'label': "
 "'employment_status'}]")


In [10]:
print(type(example))

<class 'str'>


In [11]:
import ast
# json.loads doesn't work here because it needs double quotes 
# - this only has single
le = ast.literal_eval(example)

In [12]:
print(type(le[0]))

<class 'dict'>


In [13]:
# count entities
entity_counter = Counter()
with tqdm(train_mask_feat) as pbar:
    for masks in pbar:
        masks = ast.literal_eval(masks)
        entity_counter.update([mask["label"] for mask in masks])
        
test_mask_feat = ds["test"]["spans"]
with tqdm(test_mask_feat) as pbar:
    for masks in pbar:
        masks = ast.literal_eval(masks)
        entity_counter.update([mask["label"] for mask in masks])


100%|██████████| 100000/100000 [00:10<00:00, 9575.44it/s]


In [14]:
print(f"num_entities = {len(entity_counter.items())}")

print(Fore.CYAN + "sorted entities count for dataset:")
print(Style.RESET_ALL, end="")
for i in sorted(entity_counter.items()):
    print(i)

num_entities = 55
sorted entities count for dataset:
('account_number', 26691)
('age', 11329)
('api_key', 10753)
('bank_routing_number', 15113)
('biometric_identifier', 22567)
('blood_type', 10056)
('certificate_license_number', 10922)
('city', 37660)
('company_name', 140278)
('coordinate', 16097)
('country', 51256)
('county', 14927)
('credit_debit_card', 21916)
('customer_id', 37713)
('cvv', 8397)
('date', 150997)
('date_of_birth', 30789)
('date_time', 27589)
('device_identifier', 7762)
('education_level', 20011)
('email', 103864)
('employee_id', 20945)
('employment_status', 18672)
('fax_number', 13780)
('first_name', 143434)
('gender', 11949)
('health_plan_beneficiary_number', 16903)
('http_cookie', 10678)
('ipv4', 15032)
('ipv6', 7933)
('language', 15182)
('last_name', 103558)
('license_plate', 11212)
('mac_address', 10346)
('medical_record_number', 18193)
('national_id', 6018)
('occupation', 67993)
('password', 15270)
('phone_number', 48942)
('pin', 13004)
('political_view', 10415)

In [15]:
total_len = len(ds["train"]) + len(ds["test"])
train_split = len(ds["train"]) / total_len
test_split = len(ds["test"]) / total_len
print(f"train_split = {train_split * 100:.2f}%, len = {len(ds["train"]):,}")
print(f"test_split = {test_split * 100:.2f}%, len = {len(ds["test"]):,}")

train_split = 50.00%, len = 100,000
test_split = 50.00%, len = 100,000


In [16]:
from datasets import concatenate_datasets

# move 40% of the 100k test split into train -> 140k train, 60k held out
split = ds["test"].train_test_split(test_size=0.6, seed=42)
new_train_ds = concatenate_datasets([ds["train"], split["train"]])

In [17]:
val_test = split["test"].train_test_split(test_size=0.5, seed=42)

new_ds = DatasetDict({
    "train": new_train_ds,
    "validation": val_test["train"],
    "test": val_test["test"],
})

for name, d in new_ds.items():
    print(f"{name:<11} {len(d):>7,}  {len(d) / total_len * 100:5.2f}%")

assert sum(len(d) for d in new_ds.values()) == total_len

train       140,000  70.00%
validation   30,000  15.00%
test         30,000  15.00%


In [18]:
def format_masks(batch: dict[str, list]) -> dict[str, list[list[dict]]]:
    # 12,571 spans store text unquoted (e.g. 'text': 44 for age/cvv/pin), so
    # literal_eval gives int there and arrow can't infer one struct type.
    # column named privacy_mask to match the ai4privacy dataset the trainer expects
    return {"privacy_mask": [
        [m | {"text": str(m["text"])} for m in ast.literal_eval(spans)]
        for spans in batch["spans"]
    ]}

In [19]:
# verify
fm = format_masks({"spans": [example]})
pprint(*fm["privacy_mask"])

[{'end': 8, 'label': 'first_name', 'start': 3, 'text': 'Jason'},
 {'end': 86, 'label': 'date_of_birth', 'start': 76, 'text': '1987-05-22'},
 {'end': 123,
  'label': 'street_address',
  'start': 98,
  'text': '87 Avenida De La Estrella'},
 {'end': 148, 'label': 'employment_status', 'start': 139, 'text': 'full-time'}]


In [20]:
pprint(le)

[{'end': 8, 'label': 'first_name', 'start': 3, 'text': 'Jason'},
 {'end': 86, 'label': 'date_of_birth', 'start': 76, 'text': '1987-05-22'},
 {'end': 123,
  'label': 'street_address',
  'start': 98,
  'text': '87 Avenida De La Estrella'},
 {'end': 148, 'label': 'employment_status', 'start': 139, 'text': 'full-time'}]


In [21]:
new_ds = new_ds.map(format_masks, batched=True, batch_size=10_000)

Map:   0%|          | 0/140000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

In [22]:
counters = {}
for split in new_ds:
    counters[split] = Counter()
    with tqdm(new_ds[split]["privacy_mask"]) as pbar:
        for masks in pbar:
            counters[split].update([mask["label"] for mask in masks])

100%|██████████| 30000/30000 [00:01<00:00, 23077.21it/s]


In [23]:
for name, counter in counters.items():
    print(name)
    print(f"most examples: {max(counter.items())}")
    print(f"least examples: {min(counter.items())}")

train
most examples: ('vehicle_identifier', 11184)
least examples: ('account_number', 16652)
validation
most examples: ('vehicle_identifier', 1383)
least examples: ('account_number', 4999)
test
most examples: ('vehicle_identifier', 1333)
least examples: ('account_number', 5040)


In [ ]:
# sorted so ids are deterministic
entity_classes = sorted(entity_counter)
print(entity_classes)

['account_number', 'age', 'api_key', 'bank_routing_number', 'biometric_identifier', 'blood_type', 'certificate_license_number', 'city', 'company_name', 'coordinate', 'country', 'county', 'credit_debit_card', 'customer_id', 'cvv', 'date', 'date_of_birth', 'date_time', 'device_identifier', 'education_level', 'email', 'employee_id', 'employment_status', 'fax_number', 'first_name', 'gender', 'health_plan_beneficiary_number', 'http_cookie', 'ipv4', 'ipv6', 'language', 'last_name', 'license_plate', 'mac_address', 'medical_record_number', 'national_id', 'occupation', 'password', 'phone_number', 'pin', 'political_view', 'postcode', 'race_ethnicity', 'religious_belief', 'sexuality', 'ssn', 'state', 'street_address', 'swift_bic', 'tax_id', 'time', 'unique_id', 'url', 'user_name', 'vehicle_identifier']


In [25]:
# create bio tags mapped to ids
label2id = {"O": 0}
for i, entity in enumerate(entity_classes):
    b_id = 1 + i * 2
    i_id = b_id + 1
    entity_tags = {f"B-{entity}": b_id, f"I-{entity}": i_id}
    label2id.update(entity_tags)
print(f"{len(label2id)=}")
pprint(label2id, sort_dicts=False, indent=4)

len(label2id)=111
{   'O': 0,
    'B-account_number': 1,
    'I-account_number': 2,
    'B-age': 3,
    'I-age': 4,
    'B-api_key': 5,
    'I-api_key': 6,
    'B-bank_routing_number': 7,
    'I-bank_routing_number': 8,
    'B-biometric_identifier': 9,
    'I-biometric_identifier': 10,
    'B-blood_type': 11,
    'I-blood_type': 12,
    'B-certificate_license_number': 13,
    'I-certificate_license_number': 14,
    'B-city': 15,
    'I-city': 16,
    'B-company_name': 17,
    'I-company_name': 18,
    'B-coordinate': 19,
    'I-coordinate': 20,
    'B-country': 21,
    'I-country': 22,
    'B-county': 23,
    'I-county': 24,
    'B-credit_debit_card': 25,
    'I-credit_debit_card': 26,
    'B-customer_id': 27,
    'I-customer_id': 28,
    'B-cvv': 29,
    'I-cvv': 30,
    'B-date': 31,
    'I-date': 32,
    'B-date_of_birth': 33,
    'I-date_of_birth': 34,
    'B-date_time': 35,
    'I-date_time': 36,
    'B-device_identifier': 37,
    'I-device_identifier': 38,
    'B-education_level'

In [26]:
id2label = {i:label for label, i in label2id.items()}
pprint(id2label)

{0: 'O',
 1: 'B-account_number',
 2: 'I-account_number',
 3: 'B-age',
 4: 'I-age',
 5: 'B-api_key',
 6: 'I-api_key',
 7: 'B-bank_routing_number',
 8: 'I-bank_routing_number',
 9: 'B-biometric_identifier',
 10: 'I-biometric_identifier',
 11: 'B-blood_type',
 12: 'I-blood_type',
 13: 'B-certificate_license_number',
 14: 'I-certificate_license_number',
 15: 'B-city',
 16: 'I-city',
 17: 'B-company_name',
 18: 'I-company_name',
 19: 'B-coordinate',
 20: 'I-coordinate',
 21: 'B-country',
 22: 'I-country',
 23: 'B-county',
 24: 'I-county',
 25: 'B-credit_debit_card',
 26: 'I-credit_debit_card',
 27: 'B-customer_id',
 28: 'I-customer_id',
 29: 'B-cvv',
 30: 'I-cvv',
 31: 'B-date',
 32: 'I-date',
 33: 'B-date_of_birth',
 34: 'I-date_of_birth',
 35: 'B-date_time',
 36: 'I-date_time',
 37: 'B-device_identifier',
 38: 'I-device_identifier',
 39: 'B-education_level',
 40: 'I-education_level',
 41: 'B-email',
 42: 'I-email',
 43: 'B-employee_id',
 44: 'I-employee_id',
 45: 'B-employment_status',
 4

In [27]:
all_count = Counter()
for counter in counters.values():
    all_count.update(counter)
 
assert(entity_counter == all_count)

In [28]:
# create dict to save collected data
label_info = {
    "entities": entity_classes,
    "label2id": label2id,
    "id2label": id2label,
    "all_entities_counted": dict(all_count),
    "train_counted_entities": dict(counters["train"]),
}

In [29]:
# keep only the two columns the trainer reads, named as in the ai4privacy dataset
new_ds = new_ds.rename_column("text", "source_text")
keep = ["source_text", "privacy_mask"]
new_ds = new_ds.remove_columns([c for c in new_ds["train"].column_names if c not in keep])
print(new_ds["train"].column_names)

['source_text', 'privacy_mask']


In [30]:
example = new_ds["train"][0]
pprint(example)

{'privacy_mask': [{'end': 8,
                   'label': 'first_name',
                   'start': 3,
                   'text': 'Jason'},
                  {'end': 86,
                   'label': 'date_of_birth',
                   'start': 76,
                   'text': '1987-05-22'},
                  {'end': 123,
                   'label': 'street_address',
                   'start': 98,
                   'text': '87 Avenida De La Estrella'},
                  {'end': 148,
                   'label': 'employment_status',
                   'start': 139,
                   'text': 'full-time'}],
 'source_text': 'I, Jason, am applying for a financial services account. My '
                'date of birth is 1987-05-22. I live at 87 Avenida De La '
                'Estrella. I am employed full-time.'}


In [31]:
new_ds.save_to_disk(DATASET_PATH)
# save as json
with open(LABEL_INFO_PATH, "w") as f:
    json.dump(label_info, f, indent=2)

Saving the dataset (0/1 shards):   0%|          | 0/140000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]